# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samia2310/flyrank-assignment1-week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



For this assignment, I selected the **Random Forest Classifier** as the modeling approach.

The Week-4 baseline used a simple rule-based scoring system that identified content for review using three historical signals: days since the last update, search impressions, and click-through rate (CTR). While this baseline is transparent and easy to interpret, it relies on fixed thresholds and cannot learn more complex relationships between features.

Random Forest is well suited for this problem because it can capture non-linear interactions, handle missing values after preprocessing, and provide feature importance scores for interpretation. It also performs well on structured tabular datasets such as the Content Opportunity Scoring dataset.

The model is evaluated using the same dataset and a consistent train-test split to provide a fair comparison with the Week-4 baseline.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


An 80/20 train-test split is used to evaluate the model.

Only historical features available at decision time are included. No future-derived variables or outcome-based columns are used during training, preventing data leakage.

The same dataset used for the Week-4 baseline is retained so that differences in performance are due to the modeling approach rather than differences in the data.

In [3]:
import pandas as pd

# Load dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# --------------------------
# Recreate Week-4 baseline
# --------------------------

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["low_ctr"] = (df["ctr"] < 1.0).astype(int)

df["baseline_score"] = (
    df["stale"] * 2 +
    df["visible"] * 2 +
    df["low_ctr"]
)

def action(score):
    if score >= 5:
        return "Refresh Immediately"
    elif score >= 3:
        return "Review Soon"
    else:
        return "Monitor"

df["action"] = df["baseline_score"].apply(action)

# Historical features only
features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "content_age_days",
    "search_volume",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

X = df[features].fillna(0)
y = df["action"]

print(X.shape)
X.head()

(30000, 8)


,days_since_last_update,impressions_90d,ctr,content_age_days,search_volume,avg_position,engagement_rate,scroll_rate
0,20,3803,0.76,187,10.0,10.6,5.88,4.55
1,25,15320,0.05,445,90.0,20.3,0.00,10.00
2,20,12581,0.09,141,0.0,36.5,0.00,28.57
3,22,11751,0.49,463,10.0,6.2,1.28,3.45
4,14,19140,0.13,263,0.0,44.0,0.00,24.29


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



The Random Forest model was trained using the same dataset as the Week-4 baseline.

The baseline is a transparent rule-based ranking system that recommends content refresh actions using manually selected thresholds. In contrast, the Random Forest model learns decision boundaries directly from historical observations.

The model is evaluated on the held-out test set using classification metrics. Since the Week-4 baseline produced ranked recommendations rather than classification metrics, the comparison focuses on methodology and predictive performance rather than identical numerical scores.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train model
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

# Predictions
pred = rf.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, pred)

print("Random Forest Accuracy:", round(accuracy,4))

comparison = pd.DataFrame({
    "Model":[
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Approach":[
        "Rule-based scoring",
        "Machine Learning"
    ],
    "Evaluation":[
        "Ranked action queue",
        f"Accuracy = {accuracy:.4f}"
    ]
})

comparison

Random Forest Accuracy: 0.9998


,Model,Approach,Evaluation
0,Week-4 Baseline,Rule-based scoring,Ranked action queue
1,Random Forest,Machine Learning,Accuracy = 0.9998


In [5]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance

,Feature,Importance
1,impressions_90d,0.651047
2,ctr,0.238606
6,engagement_rate,0.032224
7,scroll_rate,0.028770
0,days_since_last_update,0.025104
5,avg_position,0.015256
3,content_age_days,0.006478
4,search_volume,0.002514


In [6]:
print(classification_report(y_test, pred))

                     precision    recall  f1-score   support

            Monitor       1.00      1.00      1.00      2742
Refresh Immediately       1.00      0.75      0.86         4
        Review Soon       1.00      1.00      1.00      3254

           accuracy                           1.00      6000
          macro avg       1.00      0.92      0.95      6000
       weighted avg       1.00      1.00      1.00      6000



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



The Random Forest model successfully identifies many pages that require immediate review based on historical search performance and content freshness. However, some pages are misclassified because important factors such as content quality, seasonal traffic, search intent, and business priorities are not represented in the available features.

Feature importance shows that content freshness, impressions, click-through rate, and average search position contribute most to the predictions. Compared with the Week-4 rule-based baseline, the Random Forest model is able to learn more complex relationships among these historical signals instead of relying on fixed thresholds.

Although the model improves flexibility, its predictions should still be used as decision-support rather than fully automated decisions.

In [7]:
errors = pd.DataFrame({
    "Actual": y_test,
    "Predicted": pred
})

errors[errors["Actual"] != errors["Predicted"]].head(10)

,Actual,Predicted
3507,Refresh Immediately,Review Soon


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.